# In-Memory Vector Store

The `in_memory.py` module defines a vector-store implementation that keeps documents, embedding vectors, metadata, and identifiers in a Python dictionary.

Search operations compare query embeddings with stored embeddings using cosine similarity. The implementation supports synchronous and asynchronous ingestion, deletion, ID lookup, similarity search, metadata-aware filtering, maximal marginal relevance search, construction from text values, and JSON file persistence.

# InMemoryVectorStore

`InMemoryVectorStore` stores embedded documents in process memory.

Each entry in `store` is indexed by its document ID and contains the ID, embedding vector, page content, and metadata. Because the data is held in memory, it is not automatically preserved between processes unless `dump` and `load` are used.

## Bases

- `VectorStore`

## Attributes

1. `store`: Stores all indexed document records by document ID.

   Each nested dictionary contains `"id"`, `"vector"`, `"text"`, and `"metadata"` fields.

   * **Type:**
     ```python
     store: dict[
         str,
         dict[
             str,
             Any
         ]
     ]
     ```

2. `embedding`: Stores the embedding implementation used for documents and queries.
   * **Type:**
     ```python
     embedding: Embeddings
     ```

### Properties

1. `embeddings`: Returns the embedding implementation used by the vector store.
   * **Type:**
     ```python
     embeddings: Embeddings
     ```

### Methods

1. `__init__`: Creates an empty in-memory vector store using the supplied embedding implementation.
   * **Syntax:**
     ```python
     __init__(
         self,
         embedding: Embeddings # Embedding implementation used for documents and queries
     ) -> None
     ```

2. `delete`: Removes stored records whose IDs are included in `ids`.

   Missing IDs are ignored. When `ids` is `None` or empty, no records are removed. Additional keyword arguments are accepted for compatibility but are not used.

   * **Syntax:**
     ```python
     delete(
         self,
         ids: Sequence[str] | None = None, # IDs of records to remove
         **kwargs: Any # Additional compatibility arguments
     ) -> None
     ```

3. `adelete`: Asynchronously removes stored records.

   This implementation delegates directly to the synchronous `delete` method and performs no asynchronous I/O.

   * **Syntax:**
     ```python
     async adelete(
         self,
         ids: Sequence[str] | None = None, # IDs of records to remove
         **kwargs: Any # Additional compatibility arguments
     ) -> None
     ```

4. `add_documents`: Embeds and stores a list of documents synchronously.

   Page contents are embedded together through `embedding.embed_documents`. Explicit IDs take priority over IDs already present on the documents. When neither source provides an ID, a new UUID string is generated.

   Adding a document with an existing ID replaces the previous record stored under that ID. A `ValueError` is raised when a non-empty `ids` list has a different length from `documents`.

   * **Syntax:**
     ```python
     add_documents(
         self,
         documents: list[Document], # Documents to embed and store
         ids: list[str] | None = None, # Optional IDs overriding document IDs
         **kwargs: Any # Additional compatibility arguments
     ) -> list[str]
     ```

5. `aadd_documents`: Embeds and stores a list of documents asynchronously.

   Document embeddings are generated through `embedding.aembed_documents`. ID selection, UUID generation, record replacement, and length validation follow the same rules as `add_documents`.

   * **Syntax:**
     ```python
     async aadd_documents(
         self,
         documents: list[Document], # Documents to embed and store
         ids: list[str] | None = None, # Optional IDs overriding document IDs
         **kwargs: Any # Additional compatibility arguments
     ) -> list[str]
     ```

6. `get_by_ids`: Returns documents whose IDs exist in the store.

   Missing IDs are ignored. Returned documents preserve the requested order for IDs that are found, and each returned `Document` contains its stored ID, page content, and metadata.

   * **Syntax:**
     ```python
     get_by_ids(
         self,
         ids: Sequence[str], / # Document IDs to retrieve
     ) -> list[Document]
     ```

7. `aget_by_ids`: Asynchronously returns documents whose IDs exist in the store.

   This implementation delegates directly to `get_by_ids`.

   * **Syntax:**
     ```python
     async aget_by_ids(
         self,
         ids: Sequence[str], / # Document IDs to retrieve
     ) -> list[Document]
     ```

8. `similarity_search_with_score_by_vector`: Returns documents most similar to an embedding vector together with cosine-similarity scores.

   An optional filter is applied to reconstructed `Document` objects before similarity calculation. Results are ordered from highest to lowest similarity and limited to `k` entries.

   An empty list is returned when the store contains no matching documents.

   * **Syntax:**
     ```python
     similarity_search_with_score_by_vector(
         self,
         embedding: list[float], # Query embedding vector
         k: int = 4, # Maximum number of results
         filter: Callable[
             [Document],
             bool
         ] | None = None, # Optional document filter
         **_kwargs: Any # Additional ignored arguments
     ) -> list[
         tuple[
             Document,
             float
         ]
     ]
     ```

9. `similarity_search_with_score`: Embeds a text query and returns the most similar documents with cosine-similarity scores.

   Query embedding is performed through `embedding.embed_query`. Additional keyword arguments, including `filter`, are forwarded to `similarity_search_with_score_by_vector`.

   * **Syntax:**
     ```python
     similarity_search_with_score(
         self,
         query: str, # Query text
         k: int = 4, # Maximum number of results
         **kwargs: Any # Vector-search arguments such as filter
     ) -> list[
         tuple[
             Document,
             float
         ]
     ]
     ```

10. `asimilarity_search_with_score`: Asynchronously embeds a text query and returns similar documents with cosine-similarity scores.

    Query embedding is performed through `embedding.aembed_query`. The vector comparison itself is then performed synchronously in memory.

    * **Syntax:**
      ```python
      async asimilarity_search_with_score(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of results
          **kwargs: Any # Vector-search arguments such as filter
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

11. `similarity_search_by_vector`: Returns documents most similar to an embedding vector.

    It delegates to `similarity_search_with_score_by_vector` and discards the similarity scores.

    * **Syntax:**
      ```python
      similarity_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Maximum number of documents
          **kwargs: Any # Vector-search arguments such as filter
      ) -> list[Document]
      ```

12. `asimilarity_search_by_vector`: Asynchronously returns documents most similar to an embedding vector.

    This implementation delegates directly to the synchronous `similarity_search_by_vector` method.

    * **Syntax:**
      ```python
      async asimilarity_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Maximum number of documents
          **kwargs: Any # Vector-search arguments such as filter
      ) -> list[Document]
      ```

13. `similarity_search`: Embeds a text query and returns the most similar documents.

    It delegates to `similarity_search_with_score` and discards the cosine-similarity scores.

    * **Syntax:**
      ```python
      similarity_search(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents
          **kwargs: Any # Search arguments such as filter
      ) -> list[Document]
      ```

14. `asimilarity_search`: Asynchronously embeds a text query and returns the most similar documents.

    It delegates to `asimilarity_search_with_score` and discards the cosine-similarity scores.

    * **Syntax:**
      ```python
      async asimilarity_search(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents
          **kwargs: Any # Search arguments such as filter
      ) -> list[Document]
      ```

15. `max_marginal_relevance_search_by_vector`: Returns documents selected through maximal marginal relevance using an embedding vector.

    The method first retrieves up to `fetch_k` cosine-similar candidates, optionally after applying a document filter. It then selects up to `k` results by balancing similarity to the query against diversity among the selected documents.

    `lambda_mult=1` prioritizes similarity, while `lambda_mult=0` prioritizes diversity. An `ImportError` is raised when NumPy is unavailable.

    * **Syntax:**
      ```python
      max_marginal_relevance_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Number of documents to select
          fetch_k: int = 20, # Candidate documents considered
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          *,
          filter: Callable[
              [Document],
              bool
          ] | None = None, # Optional document filter
          **kwargs: Any # Additional compatibility arguments
      ) -> list[Document]
      ```

16. `max_marginal_relevance_search`: Embeds a text query and performs maximal marginal relevance search.

    Query embedding is generated synchronously through `embedding.embed_query`, after which selection is delegated to `max_marginal_relevance_search_by_vector`.

    * **Syntax:**
      ```python
      max_marginal_relevance_search(
          self,
          query: str, # Query text
          k: int = 4, # Number of documents to select
          fetch_k: int = 20, # Candidate documents considered
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          **kwargs: Any # Search arguments such as filter
      ) -> list[Document]
      ```

17. `amax_marginal_relevance_search`: Asynchronously embeds a text query and performs maximal marginal relevance search.

    Query embedding is generated through `embedding.aembed_query`. Candidate comparison and MMR selection are then performed synchronously in memory.

    * **Syntax:**
      ```python
      async amax_marginal_relevance_search(
          self,
          query: str, # Query text
          k: int = 4, # Number of documents to select
          fetch_k: int = 20, # Candidate documents considered
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          **kwargs: Any # Search arguments such as filter
      ) -> list[Document]
      ```

18. `from_texts`: Constructs an in-memory vector store from text values.

    A new store is initialized with the supplied embedding implementation. Text ingestion is delegated to the inherited `add_texts` interface, which ultimately stores the corresponding documents through this class.

    * **Syntax:**
      ```python
      @classmethod
      from_texts(
          cls,
          texts: list[str], # Text values used to initialize the store
          embedding: Embeddings, # Embedding implementation
          metadatas: list[
              dict[str, Any]
          ] | None = None, # Optional metadata for each text
          **kwargs: Any # Additional ingestion arguments, including optional IDs
      ) -> InMemoryVectorStore
      ```

19. `afrom_texts`: Asynchronously constructs an in-memory vector store from text values.

    A new store is initialized and the inherited asynchronous text-ingestion interface is used to embed and store the values.

    * **Syntax:**
      ```python
      @classmethod
      async afrom_texts(
          cls,
          texts: list[str], # Text values used to initialize the store
          embedding: Embeddings, # Embedding implementation
          metadatas: list[
              dict[str, Any]
          ] | None = None, # Optional metadata for each text
          **kwargs: Any # Additional ingestion arguments, including optional IDs
      ) -> InMemoryVectorStore
      ```

20. `load`: Loads a previously dumped in-memory vector store from a JSON file.

    The serialized store is deserialized through LangChain's loading utilities. A new `InMemoryVectorStore` is initialized with the supplied embedding implementation, and its `store` dictionary is replaced with the loaded data.

    File-system and deserialization errors are allowed to propagate.

    * **Syntax:**
      ```python
      @classmethod
      load(
          cls,
          path: str, # Path of the serialized vector-store file
          embedding: Embeddings, # Embedding implementation used after loading
          **kwargs: Any # Additional constructor arguments
      ) -> InMemoryVectorStore
      ```

21. `dump`: Serializes the current in-memory store to a JSON file.

    Missing parent directories are created automatically. The store is converted through LangChain's serialization utility and written using UTF-8 encoding with indentation.

    File-system and serialization errors are allowed to propagate.

    * **Syntax:**
      ```python
      dump(
          self,
          path: str # Destination JSON file path
      ) -> None
      ```

## Storage Behaviour

Each stored document uses the following internal fields:

- `"id"` contains the document identifier.
- `"vector"` contains the embedding vector.
- `"text"` contains the document page content.
- `"metadata"` contains the document metadata dictionary.

Supplying an existing ID during ingestion replaces the record currently associated with that ID.